# Feed Ranking Launch Review

Creates or updates the Feed Ranking Launch Review semantic model and report in this workspace from the GitHub repo. Run all (2-3 minutes); re-run whenever the repo changes.

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
import time
import requests
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs.report import create_report_from_reportjson, update_report_from_reportjson

REPO = "https://raw.githubusercontent.com/NirmalKumar31/social-engagement-intelligence-pro/main/powerbi"
FOLDER = "Feed_Ranking_Launch_Review"
NAME = "Feed Ranking Launch Review"
nocache = {"Cache-Control": "no-cache"}

bim = requests.get(f"{REPO}/{FOLDER}.SemanticModel/model.bim", headers=nocache, timeout=60).json()
report_json = requests.get(f"{REPO}/{FOLDER}.Report/report.json", headers=nocache, timeout=60).json()
print(f"Loaded model: {len(bim['model']['tables'])} tables; report: {len(report_json['sections'])} pages")

## 1. Semantic model (create or update)

In [ ]:
if NAME in fabric.list_datasets(mode="rest")["Dataset Name"].values:
    labs.update_semantic_model_from_bim(dataset=NAME, bim_file=bim)
else:
    labs.create_semantic_model_from_bim(dataset=NAME, bim_file=bim)
for _ in range(30):
    if NAME in fabric.list_datasets(mode="rest")["Dataset Name"].values:
        print("semantic model is ready")
        break
    time.sleep(5)

## 2. Load the data

On the first run this may fail on credentials. Set the Web source to Anonymous / Public under the semantic model's Settings → Data source credentials, then re-run this cell.

In [ ]:
labs.refresh_semantic_model(dataset=NAME)

## 3. Report (create or update)

In [ ]:
if NAME in fabric.list_reports()["Name"].values:
    update_report_from_reportjson(report=NAME, report_json=report_json)
else:
    create_report_from_reportjson(report=NAME, dataset=NAME, report_json=report_json)
print(f"Done. Open the '{NAME}' report (refresh the browser tab if it was already open).")